# Day 12: GenAI Intro, LLMs and Tokens
**Transformers, the 2026 model landscape, tokens and cost, and your first LLM API call**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas fastapi httpx

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas fastapi httpx
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. Next-token prediction and temperature
An LLM scores every candidate next token. Softmax turns scores into probabilities. Temperature reshapes them.

In [ ]:
import numpy as np
words = ["sunny", "rainy", "cloudy", "windy", "snowy"]
logits = np.array([3.0, 1.5, 1.2, 0.5, -1.0])

def softmax(x, T=1.0):
    z = np.exp((x - x.max()) / T)
    return z / z.sum()

rng = np.random.default_rng(7)
for T in [0.2, 1.0, 2.0]:
    p = softmax(logits, T)
    print(f"T={T}:", {w: round(float(v), 2) for w, v in zip(words, p)})
    print("   sample:", list(rng.choice(words, size=8, p=p)))

## 3. Attention by hand
Small made-up vectors for "bank of the river". `bank` and `river` are similar on purpose.

In [ ]:
tokens = ["bank", "of", "the", "river"]
E = np.array([[0.9, 0.1, 0.8], [0.1, 0.9, 0.1], [0.1, 0.8, 0.2], [0.8, 0.2, 0.9]])

def attention(Q, K, V):
    d = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d)
    w = np.exp(scores)
    w = w / w.sum(axis=1, keepdims=True)
    return w @ V, w

_, W = attention(E, E, E)
others = {t: round(float(W[0, i]), 2) for i, t in enumerate(tokens) if t != "bank"}
print("bank attends to:", others)
print("bank attends most to:", max(others, key=others.get))

## 4. NEW (v4): The 2026 model landscape
A simple routing table. Names come from `docs/MODEL_GUIDE.md`; confirm them against the approved model list before class.

In [ ]:
MODEL_TIERS = {   # confirm against the approved model list
    "light":   {"example": "GPT-6 Luna (if approved)", "reasoning_effort": "low"},
    "default": {"example": "GPT-6.1 Sol", "reasoning_effort": "medium"},
    "top":     {"example": "GPT-6 Astra (trainer approval)", "reasoning_effort": "high"},
    "slm":     {"example": "small open-weight model (Day 25)", "reasoning_effort": "n/a"},
}

def pick_tier(task):
    t = task.lower()
    if any(w in t for w in ["classify", "tag", "detect language"]):
        return "slm"
    if any(w in t for w in ["summarise", "extract", "rewrite"]):
        return "light"
    if any(w in t for w in ["plan", "migrate", "design", "debug"]):
        return "top" if "critical" in t else "default"
    return "default"

for task in ["classify a ticket", "summarise a call", "plan a network migration", "debug a critical outage",
             "draft a reply"]:
    tier = pick_tier(task)
    print(f"{task:<26} -> {tier:<7} {MODEL_TIERS[tier]['example']}")

**TODO:** add two SmartDesk tasks of your own and justify the tier you pick.

## 5. NEW (v4): Tokens 101
Count tokens with `tiktoken`. Compare English, Japanese, code and numbers.

In [ ]:
samples = {
    "english":  "My mobile data stopped working after the update.",
    "japanese": "アップデート後にモバイルデータが使えなくなりました。",
    "code":     "def refund(order_id: int) -> bool: return True",
    "numbers":  "Invoice 2026-11-11 total 12,345.67 JPY",
}
for name, text in samples.items():
    print(f"{name:<9} {len(text):>3} chars -> {count_tokens(text):>3} tokens")

In [ ]:
policy_page = "Customers may request a refund within 14 days of purchase. " * 60
per_page = count_tokens(policy_page)
print("Tokens per policy page:", per_page)
print("Pages that fit in a 128,000 token window:", 128_000 // per_page)

### Cost per request
Prices below are **placeholders**, not real prices. Replace them with the approved price sheet before using the numbers anywhere.

In [ ]:
PRICE_PER_1M = {"input": 1.00, "cached_input": 0.10, "output": 4.00}   # PLACEHOLDER prices

def cost(input_t, output_t, cached_t=0, reasoning_t=0):
    fresh = input_t - cached_t
    return (fresh * PRICE_PER_1M["input"] + cached_t * PRICE_PER_1M["cached_input"]
            + (output_t + reasoning_t) * PRICE_PER_1M["output"]) / 1_000_000

cases = {"short answer": (1200, 150, 0, 0), "long answer": (1200, 1200, 0, 0),
         "reasoning answer": (1200, 300, 0, 2000), "short, cached prefix": (1200, 150, 1000, 0)}
for name, args in cases.items():
    print(f"{name:<22} {cost(*args):.5f} per request | {cost(*args) * 1000:.2f} per 1,000 (placeholder units)")

## 6. Your first LLM API call
`llm()` calls the OpenAI API when a key is present, otherwise the offline stand-in. `usage()` returns the token counts of the last call.

In [ ]:
prompts = ["Summarise in one line: my bill doubled this month and nobody answers the phone.",
           "Explain a context window to a new support agent in two sentences.",
           "Give one tip to reduce token cost."]
for p in prompts:
    out = llm(p)
    u = usage()
    print(f"in={u['input_tokens']:>4} out={u['output_tokens']:>4} | {out[:90]}")

## 7. Behind FastAPI, with a token log
The key stays on the server. Every call is logged with tokens and estimated cost. We test with `TestClient`, so no server is needed.

In [ ]:
import csv, os
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI(title="SmartDesk summariser")
LOG = "token_log.csv"

class Ticket(BaseModel):
    id: int
    body: str = Field(min_length=1, max_length=4000)

@app.post("/summarise")
def summarise(t: Ticket):
    try:
        text = llm(f"Summarise this support ticket in one line: {t.body}")
    except Exception as e:                       # timeouts, rate limits
        raise HTTPException(status_code=503, detail=f"LLM unavailable: {type(e).__name__}")
    u = usage()
    row = {"ticket": t.id, **u, "est_cost": round(cost(u["input_tokens"], u["output_tokens"]), 6)}
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if new:
            w.writeheader()
        w.writerow(row)
    return {"summary": text, "usage": u}

client_api = TestClient(app)
for i, body in enumerate(["No signal at home since Monday.", "Charged twice for my November plan.",
                          "Cannot log in to the app after password reset."], 1):
    r = client_api.post("/summarise", json={"id": i, "body": body})
    print(r.status_code, r.json()["summary"][:70])
print(client_api.post("/summarise", json={"id": 9, "body": ""}).status_code, "for an empty body")
print(open(LOG).read())

**TODO (SmartDesk step 11):** add a `prompt_chars` column and a daily total cost line to the log.

## 8. Check yourself
1. Why did Japanese text use more tokens per character?
2. Which case was most expensive, and why?
3. Where is the API key in this notebook? Where is it not?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)